# Musanga V4 — NB07 v4_003 | Memory-safe ROI100 + complete-sensor inference

**Objective mapping rule:** infer only where pixel centres are at least **100 m inside the ROI polygon** and RGB, DSM, and a **single complete four-band MS flight** are valid. DSM is a coverage gate, not a model channel.

**RAM fix in v4_003:** the original block 01 materialized multiple entire aligned images and repeated WarpedVRT reads on full grids; that can exhaust Colab memory. This revision builds a lightweight **20 m diagnostic coverage preview** one **256 × 256 overview block at a time**, without ever reading an entire source image. Block 02 also reads each reference's local coverage window rather than a full map. GPU inference uses bounded 256-pixel tiles with **full-resolution** ROI and sensor checks; previews do not replace these checks. For tile discovery, candidate tiles are generated from the **ROI geometry** (not from large expanded QA pixel arrays), so thin eligible regions are not lost to overview resolution.

**Run:** 00 → 01 → 02 → 03 on CPU, then 04 on GPU. Defaults: Baego + 64 test tiles, `MAX_TILES=None` and `SITE='ALL'` for full inference. Old outputs remain in a separate v2 folder; this version writes to a new output location. The 3-fold RGB+MS ensemble is exploratory and has not been independently held-out validated.

## 00 — Input registry and reproducible objective inference settings

In [ ]:
%pip -q install rasterio geopandas pyogrio shapely scipy pandas matplotlib segmentation-models-pytorch
from pathlib import Path
from contextlib import ExitStack
from collections import defaultdict
import os,json,hashlib,gc,warnings
import numpy as np,pandas as pd,geopandas as gpd,matplotlib.pyplot as plt
import rasterio
from rasterio.windows import Window,from_bounds,transform as window_transform
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling
from rasterio.transform import Affine
from rasterio.features import rasterize
from rasterio.warp import reproject
from scipy.ndimage import distance_transform_edt
from shapely.geometry import box
from shapely.ops import unary_union
from IPython.display import display
from google.colab import drive
if not Path('/content/drive/MyDrive').exists(): drive.mount('/content/drive')
ROOT=next((p for p in (Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final'),
                        Path('/content/drive/MyDrive/Musanga/Final')) if p.is_dir()),None)
assert ROOT is not None
DATA=ROOT/'Data';V4=ROOT/'Outputs/v4';AUDIT=V4/'00_input_audit'
OLD=ROOT/'Outputs/V3_1'; NB05=V4/'05_rgb_dsm_ms_matched_v1'
MODEL_DIR=NB05/'04_fusion_models'
OUT=V4/'07_RGB_MS_100m_ROI_complete_sensors_v3'
QA_DIR=OUT/'quality_maps'; TILE_DIR=OUT/'prediction_tiles';FIG=OUT/'figures'
for p in (QA_DIR,TILE_DIR,FIG):p.mkdir(parents=True,exist_ok=True)
V31=json.loads((OLD/'00_input_audit/00_input_manifest.json').read_text())
F78=json.loads((AUDIT/'00_v4_field78_contract.json').read_text())
TRAIN=json.loads((V4/'03_paired_rgb_unet_v1/03_training_contract.json').read_text())
REG=pd.read_csv(NB05/'05_harmonized_sensor_registry.csv')
assert {'group','band','path','data_band'}.issubset(REG.columns)
MOSAICS={}
for v in V31['mosaics']:
 if str(v['file']).endswith('.tif'):
  p=Path(v['path'])
  if not p.is_file():p=DATA/'Ortho_RGB'/v['file']
  assert p.is_file(),p
  MOSAICS[v['file']]=p
assert len(MOSAICS)==6
# The exact same-flight MS registry built by NB05 (including Yoko P1/P2).
GROUP_MOSAIC={
 'Baego_MS':'Baego.tif',
 'MOS_1_2_MS':'Mos_1_2_transparent_mosaic_group1_georef.tif',
 'MOS_3_4_MS':'Mos_3_4_transparent_mosaic_group1_georef.tif',
 'MOS_5_6_MS':'Mos_5_6_transparent_mosaic_group1_georef.tif',
 'Yangambi_YBJoseph_right_MS':'Yangambi_georef.tif',
 'Yangambi_YBForgeo_right_MS':'Yangambi_georef.tif'}
for group in REG.group.unique():
 if str(group).startswith('Yoko_Carlier2020_MicaSense_'):
  yoko=[m for m in MOSAICS if 'yoko' in m.lower()]
  assert len(yoko)==1,yoko
  GROUP_MOSAIC[group]=yoko[0]
assert set(REG.group.unique())==set(GROUP_MOSAIC),(
 'MS registry changed since NB05',set(REG.group.unique())^set(GROUP_MOSAIC))
ROI=gpd.read_file(Path(V31['roi'])).to_crs('EPSG:32635')
assert ROI.crs.is_projected
ROI_FULL=unary_union(ROI.geometry)
assert ROI_FULL.is_valid and not ROI_FULL.is_empty
SITE='Baego'   # 'Baego', 'MOS', 'Yangambi', 'Yoko', 'ALL'
ROI_SETBACK_M=100.0
ROI_INTERIOR=ROI_FULL.buffer(-ROI_SETBACK_M,join_style=1)
assert not ROI_INTERIOR.is_empty,'100 m inward buffer removed the complete ROI'
QA_RES_M=20.0   # coarse diagnostic only; inference uses native RGB pixels
MAX_TILES=64    # First GPU test. Set None after checking outputs
QA_BLOCK=256    # maximum coarse QA-array dimensions per raster read
TILE=256
MIN_TILE_QA_FRACTION=0.01  # Efficiency only; not a quality threshold
PROB_THRESHOLD=.5
assert SITE in ('Baego','MOS','Yangambi','Yoko','ALL')
def site_of(m):
 return ('Baego' if m.startswith('Baego') else 'MOS' if m.startswith('Mos_')
         else 'Yangambi' if m.startswith('Yangambi') else 'Yoko')
SELECTED=[m for m in MOSAICS if SITE=='ALL' or site_of(m)==SITE]
def bands_for(group):
 q=REG[REG.group.eq(group)]
 assert len(q)==4 and set(q.band)=={'red','green','red_edge','nir'},group
 return {b:(Path(q[q.band.eq(b)].iloc[0].path),
            int(q[q.band.eq(b)].iloc[0].data_band))
         for b in ('red','green','red_edge','nir')}
for group in GROUP_MOSAIC:
 for p,_ in bands_for(group).values():assert p.is_file(),p
PAIRS=pd.read_csv(AUDIT/'00_rgb_dsm_pair_audit.csv')
DSM_BY_MOSAIC={Path(z.rgb_path).name:Path(z.dsm_path) for z in PAIRS.itertuples(index=False)}
assert set(MOSAICS)==set(DSM_BY_MOSAIC)
assert all(p.is_file() for p in DSM_BY_MOSAIC.values())
print('Mosaics:',SELECTED,'| ROI setback:',ROI_SETBACK_M,'m | outputs:',OUT)


## 01 — Memory-safe, blockwise coverage overview (CPU)

Reads only a 256×256 pixel overview block per sensor at a time. Source rasters remain on disk; even large mosaics do not require huge arrays. The 20 m overview is **diagnostic only**. At inference, the 100 m ROI interior and each required sensor are checked again at **native RGB resolution**.

In [ ]:
from rasterio.warp import reproject
from rasterio.windows import Window
from rasterio.enums import ColorInterp
def grid_setup(mosaic):
 with rasterio.open(MOSAICS[mosaic]) as rgb:
  assert rgb.crs is not None
  interior=gpd.GeoSeries([ROI_INTERIOR],crs='EPSG:32635').to_crs(rgb.crs).iloc[0]
  geom=interior.intersection(box(*rgb.bounds))
  if geom.is_empty:return None
  xmin,ymin,xmax,ymax=geom.bounds
  left=np.floor(xmin/QA_RES_M)*QA_RES_M
  top=np.ceil(ymax/QA_RES_M)*QA_RES_M
  width=int(np.ceil((xmax-left)/QA_RES_M))
  height=int(np.ceil((top-ymin)/QA_RES_M))
  assert 0<width and 0<height
  transform=Affine(QA_RES_M,0,left,0,-QA_RES_M,top)
  return (height,width),transform,rgb.crs,geom
def iter_blocks(shape,step=QA_BLOCK):
 height,width=shape
 for row in range(0,height,step):
  for col in range(0,width,step):
   yield Window(col,row,min(step,width-col),min(step,height-row))
def valid_read(vrt,index,win,positive=False):
 a=vrt.read(index,window=win,masked=True,out_dtype='float32')
 x=np.asarray(a.filled(np.nan),np.float32)
 ok=(~np.ma.getmaskarray(a))&np.isfinite(x)
 if positive:ok&=(x>0)
 return ok
def source_vrts(stack,group,crs,transform,shape):
 out={}
 for band,(p,index) in bands_for(group).items():
  src=stack.enter_context(rasterio.open(p))
  vrt=stack.enter_context(WarpedVRT(src,crs=crs,transform=transform,
        width=shape[1],height=shape[0],resampling=Resampling.bilinear))
  out[band]=(vrt,index)
 return out
def audit_one(mosaic):
 setup=grid_setup(mosaic)
 if setup is None:
  return dict(mosaic=mosaic,qa_path='',roi_interior_pixels=0,
              rgb_available_fraction=0.,dsm_available_fraction=0.,
              ms_available_fraction=0.,accepted_fraction=0.)
 shape,tr,crs,roi_geom=setup
 dest=QA_DIR/(mosaic.replace('.tif','')+'_objective_sensor_support_100m.tif')
 profile=dict(driver='GTiff',height=shape[0],width=shape[1],
              count=1,dtype='uint8',crs=crs,transform=tr,
              nodata=0,compress='deflate',tiled=True)
 totals=dict(inside=0,rgb=0,dsm=0,ms=0,good=0)
 with ExitStack() as stack:
  src_rgb=stack.enter_context(rasterio.open(MOSAICS[mosaic]))
  rgb=stack.enter_context(WarpedVRT(src_rgb,crs=crs,transform=tr,
         width=shape[1],height=shape[0],resampling=Resampling.bilinear))
  src_dsm=stack.enter_context(rasterio.open(DSM_BY_MOSAIC[mosaic]))
  dsm=stack.enter_context(WarpedVRT(src_dsm,crs=crs,transform=tr,
         width=shape[1],height=shape[0],resampling=Resampling.bilinear))
  groups={group:source_vrts(stack,group,crs,tr,shape)
          for group,owner in GROUP_MOSAIC.items() if owner==mosaic}
  assert groups,('Missing MS group',mosaic)
  with rasterio.open(dest,'w',**profile) as output:
   n_blocks=0
   for win in iter_blocks(shape):
    h,w=int(win.height),int(win.width)
    # Bounding-box rejection before any band read.
    rect=box(*rasterio.windows.bounds(win,tr))
    if not roi_geom.intersects(rect):
     output.write(np.zeros((h,w),np.uint8),1,window=win)
     continue
    roi_mask=rasterize([(roi_geom.intersection(rect),1)],
       out_shape=(h,w),transform=rasterio.windows.transform(win,tr),
       fill=0,dtype='uint8').astype(bool)
    valid_rgb=np.logical_and.reduce([
       valid_read(rgb,k,win) for k in (1,2,3)])
    if src_rgb.count>=4 and src_rgb.colorinterp[3]==ColorInterp.alpha:
     alpha=valid_read(rgb,4,win,positive=True)
     valid_rgb&=alpha
    valid_dsm=valid_read(dsm,1,win)
    ms_by_flight=[]
    for name,flight in groups.items():
     valid=np.logical_and.reduce([valid_read(vrt,idx,win,positive=True)
                          for vrt,idx in flight.values()])
     ms_by_flight.append(valid)
    valid_ms=np.logical_or.reduce(ms_by_flight)
    good=roi_mask&valid_rgb&valid_dsm&valid_ms
    mask=np.zeros((h,w),np.uint8)
    mask[roi_mask&valid_rgb]=1
    mask[roi_mask&valid_rgb&valid_dsm]=2
    mask[good]=3
    output.write(mask,1,window=win)
    for key,m in [('inside',roi_mask),('rgb',roi_mask&valid_rgb),
                  ('dsm',roi_mask&valid_dsm),('ms',roi_mask&valid_ms),
                  ('good',good)]:
     totals[key]+=int(m.sum())
    n_blocks+=1
   print(mosaic,'processed',n_blocks,'in-ROI overview blocks',flush=True)
 count=max(1,totals['inside'])
 with rasterio.open(dest) as src:
  # Read only a small display image; never load full QA raster for plotting.
  scale=max(1,math.ceil(max(src.width,src.height)/1000))
  display_arr=src.read(1,out_shape=(
      max(1,src.height//scale),max(1,src.width//scale)),
      resampling=Resampling.nearest)
 fig,ax=plt.subplots(1,1,figsize=(8,5))
 im=ax.imshow(display_arr,cmap='viridis',vmin=0,vmax=3,interpolation='nearest')
 ax.set_title(f'{mosaic}: full-sensor overview ({totals["good"]/count:.1%} within ROI interior)')
 ax.axis('off');fig.colorbar(im,ax=ax,shrink=.7,label='0=outside/missing; 3=all sensors')
 fig.savefig(FIG/(mosaic.replace('.tif','')+'_objective_sensor_support.png'),
             dpi=140,bbox_inches='tight')
 plt.show();plt.close(fig)
 return dict(mosaic=mosaic,qa_path=str(dest),
      roi_interior_pixels=totals['inside'],
      rgb_available_fraction=totals['rgb']/count,
      dsm_available_fraction=totals['dsm']/count,
      ms_available_fraction=totals['ms']/count,
      accepted_fraction=totals['good']/count)
QA=pd.DataFrame([audit_one(m) for m in SELECTED])
QA.to_csv(OUT/'01_objective_sensor_coverage.csv',index=False)
display(QA.round(3))
print('Overview completed with blockwise reads. Native-resolution checks occur in Section 04.')


## 02 — Sensor coverage within reference footprints (local-window reads)

This table is descriptive only. It does not affect inference or reinterpret NB05's held-out validation.

In [ ]:
UNITS=gpd.read_file(V4/'01_spatial_cover_folds_v2/01_reference_groups_and_folds_v2.gpkg')
UNITS.unit_id=UNITS.unit_id.astype(str)
OWNERS=pd.read_csv(V4/'02_rgb_circles_plus_plots_v3/02_reference_mosaic_owners_v2.csv')
OWNERS.unit_id=OWNERS.unit_id.astype(str)
UNITS=UNITS.merge(
 OWNERS[['unit_type','unit_id','mosaic','roi_mosaic_coverage_fraction']],
 on=['unit_type','unit_id'],validate='one_to_one')
UNITS=UNITS[UNITS.roi_mosaic_coverage_fraction.ge(.95)]
assert len(UNITS)==106
coverage=[]
for mosaic in SELECTED:
 file=QA.loc[QA.mosaic.eq(mosaic),'qa_path'].iloc[0]
 if not file or not Path(file).is_file():continue
 with rasterio.open(file) as ds:
  for z in UNITS[UNITS.mosaic.eq(mosaic)].to_crs(ds.crs).itertuples():
   raw=rasterio.windows.from_bounds(*z.geometry.bounds,transform=ds.transform)
   row0=max(0,int(np.floor(raw.row_off)))
   col0=max(0,int(np.floor(raw.col_off)))
   row1=min(ds.height,int(np.ceil(raw.row_off+raw.height)))
   col1=min(ds.width,int(np.ceil(raw.col_off+raw.width)))
   if row1<=row0 or col1<=col0:continue
   window=Window(col0,row0,col1-col0,row1-row0)
   status=ds.read(1,window=window)
   footprint=rasterize([(z.geometry,1)],out_shape=status.shape,
      transform=ds.window_transform(window),fill=0,dtype='uint8').astype(bool)
   if not footprint.any():continue
   coverage.append(dict(site=z.site,unit_type=z.unit_type,unit_id=z.unit_id,
      mosaic=mosaic,all_sensor_fraction=float((status[footprint]==3).mean()),
      ROI_or_RGB_unavailable=float((status[footprint]==0).mean()),
      RGB_only_fraction=float((status[footprint]==1).mean()),
      RGB_DSM_without_MS=float((status[footprint]==2).mean())))
REFERENCE_QA=pd.DataFrame(coverage)
REFERENCE_QA.to_csv(OUT/'02_reference_sensor_coverage.csv',index=False)
if len(REFERENCE_QA):
 display(REFERENCE_QA.sort_values('all_sensor_fraction').head(20).round(3))
print('Footprints audited:',len(REFERENCE_QA))


## 03 — Deterministic readiness check (no manual quality approval)

Automatic inference is permitted only on pixels inside the inward-buffered ROI and with all sensor measurements available. No visual quality classification is used.

In [ ]:
assert len(QA)==len(SELECTED)
assert all(Path(v).is_file() for v in QA.qa_path if isinstance(v,str) and v)
assert QA.roi_interior_pixels.ge(0).all()
assert QA.accepted_fraction.between(0,1).all()
print('Objective ROI setback:',ROI_SETBACK_M,'metres')
print('Ready for GPU inference. Only complete RGB + DSM + 4-band MS pixels will be written.')
display(QA[['mosaic','accepted_fraction']])


## 04 — Inference using frozen RGB+MS fold ensemble, only where **all sensors** are valid (GPU)

The 2 m QA raster is only for locating candidate tiles. At full RGB resolution, independently verify **ROI interior, valid RGB, valid DSM and all four MS bands from one acquisition**. If two complete MS acquisitions overlap, select the acquisition supplying the most fully supported pixels *for the tile*, without mixing bands. Output NoData outside the approved objective domain. Use 64 tiles for the initial smoke test, then set MAX_TILES=None for all selected mosaics.

Note: The RGB+MS network has 7 channels; **DSM is a required availability gate, not a network input**. The ensemble averages three NB05 fold-trained models. Interpret the map as exploratory inference, not a new held-out validation.

In [ ]:
import torch
import segmentation_models_pytorch as smp
from rasterio.windows import bounds as window_bounds
assert torch.cuda.is_available(),'Select a GPU Colab runtime and rerun 00–03'
DEVICE='cuda'
def rgb_model_A(fold):
 nb03=V4/'03_paired_rgb_unet_v1'
 ck=torch.load(nb03/f'fold{fold}_A_{TRAIN["training_key"][:18]}.pt',
               map_location='cpu',weights_only=False)
 assert ck['training_key']==TRAIN['training_key']
 net=smp.Unet(encoder_name='resnet18',encoder_weights=None,in_channels=3,classes=1)
 net.load_state_dict({k.removeprefix('net.'):v for k,v in ck['model'].items()})
 return net
def ms_net(fold):
 base=rgb_model_A(fold).state_dict()
 net=smp.Unet(encoder_name='resnet18',encoder_weights=None,in_channels=7,classes=1)
 state=net.state_dict()
 for k,v in state.items():
  if k=='encoder.conv1.weight':
   v[:,:3]=base[k];v[:,3:]=0
  else:v.copy_(base[k])
 net.load_state_dict(state)
 ck=torch.load(MODEL_DIR/f'fold{fold}_RGB_MS_v1.pt',
               map_location='cpu',weights_only=False)
 assert ck['complete'] and ck['baseline_training_key']==TRAIN['training_key']
 net.load_state_dict(ck['state'],strict=True)
 return net.to(DEVICE).eval(),ck['fingerprint']
MODELS=[];MODEL_HASH=[]
for fold in range(3):
 model,h=ms_net(fold)
 MODELS.append(model);MODEL_HASH.append(str(h))
MODEL_SIGNATURE=hashlib.sha256(json.dumps(MODEL_HASH).encode()).hexdigest()[:12]
QA_SIGNATURE=hashlib.sha256(json.dumps(dict(
 schema='objective_ROI100_RGB_DSM_MS_v2',setback=ROI_SETBACK_M,
 resolution=QA_RES_M,required=('RGB','DSM','red','green','red_edge','nir'),
 model=MODEL_SIGNATURE,registry=[
   (str(z.path),int(z.data_band),Path(z.path).stat().st_size,Path(z.path).stat().st_mtime_ns)
   for z in REG.itertuples()],
 qa_files=[(str(z.qa_path),Path(z.qa_path).stat().st_size)
           for z in QA.itertuples() if str(z.qa_path)]),
 sort_keys=True).encode()).hexdigest()[:12]
print('NB05 RGB+MS model ensemble:',MODEL_SIGNATURE,'| coverage policy:',QA_SIGNATURE)
def make_vrts(ds,group,stack):
 result={}
 for b,(p,index) in bands_for(group).items():
  src=stack.enter_context(rasterio.open(p))
  vrt=stack.enter_context(WarpedVRT(src,crs=ds.crs,transform=ds.transform,
       width=ds.width,height=ds.height,resampling=Resampling.bilinear))
  result[b]=(vrt,index)
 return result
def read_ms_window(vrts,win):
 layers=[];valid=[]
 for b in ('red','green','red_edge','nir'):
  vrt,index=vrts[b]
  a=vrt.read(index,window=win,masked=True,out_dtype='float32')
  v=np.asarray(a.filled(0),np.float32)
  ok=(~np.ma.getmaskarray(a))&np.isfinite(v)&(v>0)
  layers.append(np.where(ok,v,0.));valid.append(ok)
 raw=np.stack(layers)
 valid=np.logical_and.reduce(valid)
 relative=np.where(valid[None],
        raw/np.maximum(raw.sum(axis=0,keepdims=True),1e-8),0.)
 return relative.astype('float32'),valid
def rgb_window(ds,win):
 a=ds.read([1,2,3],window=win,masked=True)
 assert all(np.dtype(t)==np.dtype('uint8') for t in ds.dtypes[:3])
 valid=(~np.ma.getmaskarray(a).any(axis=0))
 if ds.count>=4 and ds.colorinterp[3].name=='alpha':
  alpha=ds.read(4,window=win,masked=False)
  valid&=np.asarray(alpha)>0
 rgb=np.asarray(a.filled(0),np.float32)/255.
 return rgb,valid
def dsm_window(vrt,win):
 a=vrt.read(1,window=win,masked=True,out_dtype='float32')
 z=np.asarray(a.filled(np.nan),np.float32)
 return (~np.ma.getmaskarray(a))&np.isfinite(z)
def roi_window(ds,win,interior_geom):
 clip=interior_geom.intersection(box(*window_bounds(win,ds.transform)))
 if clip.is_empty:return np.zeros((int(win.height),int(win.width)),bool)
 return rasterize([(clip,1)],out_shape=(int(win.height),int(win.width)),
                  transform=ds.window_transform(win),fill=0,
                  dtype='uint8').astype(bool)
def tiles_for_qa(ds,qa,interior):
 # Geometric tile discovery: no giant arrays, no missed narrow strips due
 # to a coarse sensor overview. Iterate only the inward ROI bounding range.
 if interior.is_empty:return []
 bb=rasterio.windows.from_bounds(*interior.bounds,transform=ds.transform)
 row0=max(0,int(np.floor(bb.row_off))//TILE)
 col0=max(0,int(np.floor(bb.col_off))//TILE)
 row1=min((ds.height-1)//TILE,int(np.ceil(bb.row_off+bb.height))//TILE)
 col1=min((ds.width-1)//TILE,int(np.ceil(bb.col_off+bb.width))//TILE)
 # Intersect ROI per tile, then check actual RGB/DSM/MS at full resolution
 # in run_mosaic. QA overview is not used for tile eligibility.
 return ((row,col) for row in range(row0,row1+1)
         for col in range(col0,col1+1)
         if interior.intersects(box(*rasterio.windows.bounds(
            Window(col*TILE,row*TILE,
              min(TILE,ds.width-col*TILE),min(TILE,ds.height-row*TILE)),
            ds.transform))))
@torch.inference_mode()
def run_mosaic(mosaic):
 qa_rec=QA.set_index('mosaic').loc[mosaic]
 if not qa_rec.qa_path or not Path(qa_rec.qa_path).is_file():return []
 manifest=[]
 mdir=TILE_DIR/mosaic.replace('.tif','')
 mdir.mkdir(parents=True,exist_ok=True)
 saved=0
 with ExitStack() as stack:
  rgb=stack.enter_context(rasterio.open(MOSAICS[mosaic]))
  qa=stack.enter_context(rasterio.open(qa_rec.qa_path))
  dsm=stack.enter_context(rasterio.open(DSM_BY_MOSAIC[mosaic]))
  dsm_vrt=stack.enter_context(WarpedVRT(dsm,crs=rgb.crs,
       transform=rgb.transform,width=rgb.width,height=rgb.height,
       resampling=Resampling.bilinear))
  interior=gpd.GeoSeries([ROI_INTERIOR],crs='EPSG:32635').to_crs(rgb.crs).iloc[0]
  flights={group:make_vrts(rgb,group,stack) for group,owner in GROUP_MOSAIC.items()
           if owner==mosaic}
  locations=tiles_for_qa(rgb,qa,interior)
  print(mosaic,'streaming candidate tiles inside 100m ROI',flush=True)
  for row,col in locations:
   h=min(TILE,rgb.height-row*TILE);w=min(TILE,rgb.width-col*TILE)
   win=Window(col*TILE,row*TILE,w,h)
   inside=roi_window(rgb,win,interior)
   if not inside.any():continue
   dest=mdir/f'qa{QA_SIGNATURE}_m{MODEL_SIGNATURE}_r{row:05d}_c{col:05d}.tif'
   if dest.is_file():
    manifest.append(dict(mosaic=mosaic,row=row,col=col,path=str(dest),
                         status='reused',sensor_flight='cached'))
    saved+=1
    if MAX_TILES is not None and saved>=MAX_TILES:break
    continue
   x,rgbok=rgb_window(rgb,win)
   dsmok=dsm_window(dsm_vrt,win)
   basic=inside&rgbok&dsmok
   if not basic.any():continue
   candidates=[]
   for name,vrts in flights.items():
    ms,msok=read_ms_window(vrts,win)
    candidates.append((int((basic&msok).sum()),name,ms,msok))
   nvalid,flight,ms,msok=max(candidates,key=lambda t:t[0])
   valid=basic&msok
   if not nvalid or float(valid.mean())<MIN_TILE_QA_FRACTION:continue
   features=np.zeros((7,TILE,TILE),np.float32)
   features[:3,:h,:w]=x
   features[3:,:h,:w]=ms
   # Model inputs match NB05: RGB / 255, and 4 same-flight relative MS bands.
   tensor=torch.from_numpy(features[None]).to(DEVICE)
   predictions=[]
   for model in MODELS:
    with torch.autocast('cuda'):
     prob=torch.sigmoid(model(tensor)[:,0]).float()[0].cpu().numpy()[:h,:w]
    predictions.append(prob)
   probability=np.mean(predictions,axis=0).astype('float32')
   probability[~valid]=-9999.
   binary=np.where(valid,(probability>=PROB_THRESHOLD).astype('float32'),-9999.)
   profile=rgb.profile.copy()
   profile.update(driver='GTiff',count=2,height=h,width=w,
      transform=rgb.window_transform(win),dtype='float32',
      nodata=-9999.,compress='deflate',tiled=False)
   tmp=dest.with_name(dest.stem+'.temp.tif')
   with rasterio.open(tmp,'w',**profile) as dst:
    dst.write(probability,1);dst.write(binary,2)
    dst.set_band_description(1,'Musanga probability — RGB DSM 4-band MS + ROI100')
    dst.set_band_description(2,'Musanga binary threshold 0.5 — valid sensors only')
    dst.update_tags(ms_flight=flight,roi_setback_m=ROI_SETBACK_M,
                    model_folds='0,1,2',qa_signature=QA_SIGNATURE)
   os.replace(tmp,dest)
   manifest.append(dict(mosaic=mosaic,row=row,col=col,path=str(dest),
              status='new',sensor_flight=flight,
              valid_fraction=float(valid.mean())))
   saved+=1
   if saved%10==0:print('Completed',mosaic,saved,'tiles',flush=True)
   if MAX_TILES is not None and saved>=MAX_TILES:break
 return manifest
TILE_MANIFEST=[]
for mosaic in SELECTED:TILE_MANIFEST.extend(run_mosaic(mosaic))
TILE_MANIFEST=pd.DataFrame(TILE_MANIFEST)
TILE_MANIFEST.to_csv(OUT/'04_prediction_tile_manifest.csv',index=False)
print('Inference tiles saved/reused:',len(TILE_MANIFEST))
if len(TILE_MANIFEST):
 display(TILE_MANIFEST.groupby(['mosaic','status']).size().rename('tiles').reset_index())
else:print('No eligible tiles. Check ROI interior and source coverage tables.')


## 05 — Example figures from actual inference: RGB, MS, reference annotations, probability, binary prediction and overlay

Only use genuine predictions from the current 100 m ROI-interior run. Selected examples must be fully covered by saved tiles (the 64-tile pilot may not yet intersect reference footprints). Source imagery is for interpretation; don't confuse in-sample ensemble inference with held-out evaluation.

In [ ]:
import matplotlib.colors as mcolors
from rasterio.merge import merge
# To keep figures honest, use only units actually intersecting saved prediction
# tiles and require >=50% quality-approved support within the shown unit.
assert 'TILE_MANIFEST' in globals() and len(TILE_MANIFEST)>0, 'Run Section 04 first. No accepted inference tiles exist.'
PLOT_SOURCE=gpd.read_file(F78['exhaustive_plot_crowns']).to_crs('EPSG:32635')
PLOT_SOURCE['positive']=pd.to_numeric(PLOT_SOURCE.Class_ID,errors='coerce').eq(1)
CIRCLE_SOURCE=gpd.read_file(F78['circle_positive_crowns']).to_crs('EPSG:32635')
CIRCLE_SOURCE['positive']=True
def plot_a_few():
 used=[]
 for mosaic,files in TILE_MANIFEST.groupby('mosaic'):
  units=UNITS[UNITS.mosaic.eq(mosaic)].copy()
  if not len(units):continue
  tilefiles=list(files.path)
  # Reference footprints can straddle tile boundaries. Select one footprint
  # whose whole buffered bounding box intersects saved tiles; mosaic merge
  # below reconstructs an excerpt from the georeferenced probability tiles.
  with ExitStack() as stack:
   src=[stack.enter_context(rasterio.open(p)) for p in tilefiles]
   union_bounds=unary_union([box(*p.bounds) for p in src])
   with rasterio.open(MOSAICS[mosaic]) as rgb:
    with ExitStack() as vst:
     group=next((g for g,m in GROUP_MOSAIC.items() if m==mosaic),None)
     vrts=make_vrts(rgb,group,vst)
     for z in units.to_crs(rgb.crs).itertuples():
      if not union_bounds.covers(z.geometry):continue
      win=from_bounds(*z.geometry.bounds,transform=rgb.transform)
      c0=max(0,int(np.floor(win.col_off)));r0=max(0,int(np.floor(win.row_off)))
      c1=min(rgb.width,int(np.ceil(win.col_off+win.width)))
      r1=min(rgb.height,int(np.ceil(win.row_off+win.height)))
      ww=c1-c0;hh=r1-r0
      if ww<15 or hh<15 or ww*hh>8_000_000:continue
      window=Window(c0,r0,ww,hh)
      ptiles=[q for q in src if box(*q.bounds).intersects(z.geometry)]
      if not ptiles:continue
      pred,tr=merge(ptiles,bounds=rasterio.windows.bounds(window,rgb.transform),
                    nodata=-9999.,indexes=[1],res=(abs(rgb.transform.a),abs(rgb.transform.e)))
      p=np.asarray(pred[0],float)
      if p.shape!=(hh,ww):continue
      refmask=rasterize([(z.geometry,1)],out_shape=(hh,ww),
               transform=rgb.window_transform(window),fill=0).astype(bool)
      good=(p>=0)&(p<=1)
      if not refmask.any() or good[refmask].mean()<.5:continue
      rgb3,rv=rgb_window(rgb,window)
      ms,msv=read_ms_window(vrts,window)
      false=np.stack([ms[3],ms[2],ms[0]],axis=-1)
      validMS=msv&(ms.sum(axis=0)>0)
      if validMS.any():
       for band in range(3):
        q=false[:,:,band]
        lo,hi=np.percentile(q[validMS],[2,98])
        false[:,:,band]=np.clip((q-lo)/max(hi-lo,1e-8),0,1)
      false[~validMS]=1
      extent=rasterio.windows.bounds(window,rgb.transform)
      extent=[extent[0],extent[2],extent[1],extent[3]]
      fig,axes=plt.subplots(1,6,figsize=(21,3.7),layout='constrained')
      axes[0].imshow(np.moveaxis(rgb3,0,-1),extent=extent,origin='upper')
      axes[1].imshow(false,extent=extent,origin='upper')
      axes[2].imshow(np.moveaxis(rgb3,0,-1),extent=extent,origin='upper')
      src_gdf=CIRCLE_SOURCE if z.unit_type=='circle' else PLOT_SOURCE
      selected=src_gdf.iloc[src_gdf.sindex.query(
          gpd.GeoSeries([z.geometry],crs=rgb.crs).to_crs('EPSG:32635').iloc[0],
          predicate='intersects')].copy()
      if len(selected):
       selected=selected.to_crs(rgb.crs)
       for positive,color in [(True,'#ff5544'),(False,'#39c8ed')]:
        part=selected[selected.positive.eq(positive)]
        if len(part):part.boundary.plot(ax=axes[2],color=color,linewidth=.8)
      probability=np.ma.masked_where(~good,p)
      axes[3].imshow(probability,extent=extent,origin='upper',cmap='magma',vmin=0,vmax=1)
      axes[4].imshow(np.ma.masked_where(~good,p>=PROB_THRESHOLD),
                    extent=extent,origin='upper',cmap='Greens',vmin=0,vmax=1)
      axes[5].imshow(np.moveaxis(rgb3,0,-1),extent=extent,origin='upper')
      overlay=np.ma.masked_where(~good|(p<PROB_THRESHOLD),p)
      axes[5].imshow(overlay,extent=extent,origin='upper',cmap='spring',vmin=0,vmax=1,alpha=.50)
      for ax,title in zip(axes,['RGB','MS NIR/RE/R','Annotations','Probability',
                                 'Binary Musanga','Prediction overlay']):
       gpd.GeoSeries([z.geometry],crs=rgb.crs).boundary.plot(
          ax=ax,color='#ffec4d',linewidth=1.4)
       ax.set(xlim=(extent[0],extent[1]),ylim=(extent[2],extent[3]),title=title)
       ax.set_xticks([]);ax.set_yticks([])
      fig.suptitle(f'{z.site} {z.unit_type} {z.unit_id} | mapped MS quality only')
      out=FIG/f'example_{mosaic.replace(".tif","")}_{z.unit_type}_{z.unit_id}.png'
      fig.savefig(out,dpi=175,bbox_inches='tight');plt.show();plt.close(fig)
      used.append(dict(site=z.site,unit_type=z.unit_type,unit_id=z.unit_id,
                       path=str(out),approved_fraction=float(good[refmask].mean())))
      if len(used)>=6:return used
 return used
EXAMPLES=pd.DataFrame(plot_a_few())
EXAMPLES.to_csv(OUT/'05_example_atlas_manifest.csv',index=False)
print('Completed QA example panels:',len(EXAMPLES))
display(EXAMPLES)
if not len(EXAMPLES):print('Smoke-test tiles did not fully cover any reference footprint. '
                            'Expand MAX_TILES or choose a reference-centered tile set.')
